| Middleware | Parameters | Main purpose |
|---|---|---|
| `before_agent` | `state, runtime` | Before the entire agent starts |
| `before_model` | `state, runtime` | Before each model call |
| `after_model` | `state, runtime` | After each model call |
| `after_agent` | `state, runtime` | After the entire agent finishes |
| `wrap_model_call` | `request, handler` | Intercept/modify the actual model call |
| `wrap_tool_call` | `request, handler` | Intercept/modify tool execution |

                                    wrap_model_call starts
                                              |
                                Your code before handler(request)
                                              |
                                Modify the prompt, model, or tools
                                              |
                                       handler(request)
                                              |
                                The actual downstream model call
                                              |
                                Your code after handler(request)
                                              |
                                Inspect the response or log it
                                              |
                                     wrap_model_call ends

When you use middleware like before_model(state, runtime), the state parameter contains the agent's current state. When you return a dictionary, you're telling LangChain which parts of that state to update.

```py
@before_model
def my_middleware(state, runtime):
    print(state["messages"])  # Read existing messages

    return {
        "status": "thinking"
    }

    # or 

    return {
    "messages": [
        HumanMessage(content="Keep the explanation simple.")
    ]
}
```

```py
# use before middleware it accepts state and runtime
@before_model
def check_user(state, runtime):

    plan = runtime.context["plan"]

    if plan == "free":
        # modify/limit behavior based on plan
        ...

    if state["current_task"] == "expensive_task":
        # modify state or control execution
        ...

    return None

*************************************************************
# use wrap model to get agent informations from request

from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def my_middleware(request, handler):
    print("Before model")

    new_request = request.override(
    system_message=SystemMessage(content="You are a Python tutor."),
    model=another_model,
    tools=selected_tools,
)
    response = handler(new_request)

    print("After model")

    return response

request
   │
   ├── request.model
   ├── request.messages
   ├── request.system_message
   ├── request.tools
   ├── request.state
   ├── request.runtime
   ├── request.model_settings
   └── request.override()

handler
   │
   └── actually calls the next model layer

**************************************************************
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

class UserContext(TypedDict):
    user_id: str
    plan: str

@dynamic_prompt
def add_user_context(request: ModelRequest) -> str:
    context = request.runtime.context

    return (
        "You are a helpful assistant.\n"
        f"Current user ID: {context['user_id']}\n"
        f"Current user's plan: {context['plan']}"
    )

agent = create_agent(
    model=model,
    context_schema=UserContext,
    middleware=[add_user_context],
)

res = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What's user 123's plan?"}
        ]
    },
    context={"user_id": "123", "plan": "free"},
)

print(res["messages"][-1].content)

```

In [1]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="medium",
)


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,  
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 6),
            keep=("messages", 2),
        )
    ],
)

In [3]:

from pprint import pprint

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    # Mock sending: nothing is actually sent.
    print(f"To: {to}\nSubject: {subject}\n\n{body}")
    return "Email sent successfully."


agent = create_agent(
    model=model,  
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

config = {
    "configurable": {"thread_id": "email-001"}
}

# 1. Ask the agent to draft and send an email.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "write email to my boss about the new project cosmos like the project is done and sned it to tester@exemple.com"
                ),
            }
        ]
    },
    config=config,
)

pprint(result)


{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi,\n'
                                                                          '\n'
                                                                          'I '
                                                                          'wanted '
                                                                          'to '
                                                                          'update '
                                                                          'you '
                                                                          'that '
                                                                          'the '
                                                                          'new '
                                                                          'project '
                                                                          'Cosmos '
                  

In [4]:
    # 3. After reviewing it, approve the proposed send.
result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "approve"}]
            }
        ),
        config=config,
    )

print(result)


To: tester@exemple.com
Subject: Project Cosmos - Completion Update

Hi,

I wanted to update you that the new project Cosmos is now complete.

All deliverables have been finalized and the project is ready for review and handover. Please let me know if you would like a summary walkthrough or if there are any final approvals needed before we move to the next phase.

Best regards
{'messages': [HumanMessage(content='write email to my boss about the new project cosmos like the project is done and sned it to tester@exemple.com', additional_kwargs={}, response_metadata={}, id='0639c16b-b4d8-491a-ad4d-87cbd7b81f17'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 331, 'prompt_tokens': 444, 'total_tokens': 775, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'me

In [8]:
from langchain.messages import ToolMessage

for message in result["messages"]:
    if isinstance(message, ToolMessage):
        print(message.content)

Email sent successfully.
